In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [2]:
import sys
sys.path.append('/g/data/rd53/wy2165/disequilibrium/code_analysis/')
from Lowess_fit import run_lowess_from_csv

In [3]:
region = 19
y_col = 'mass_remaining'
outpath = Path('/g/data/rd53/wy2165/disequilibrium/pygem_oggm/')

In [4]:
tag = 'median_a'
ds = xr.open_dataset(outpath / f'PyGEM_global_glacier_stats_{tag}.nc')

In [5]:
missing =  pd.read_csv(f'/scratch/k10/wy2165/PyGEM/frontalablation_data/analysis/{region}-frontalablation_cal_ind-missing.csv')
ind = pd.read_csv(f'/scratch/k10/wy2165/PyGEM/frontalablation_data/analysis/{region}-frontalablation_cal_ind.csv')

### Mass

In [6]:
def summarize_by_class(ds, class_coord):
    static_by_class = (
        xr.Dataset({
            'n_glaciers': ds['rgi_area_km2'].notnull(),
            'rgi_area_km2': ds['rgi_area_km2'],
            'vol_itmix_m3': ds['vol_itmix_m3'],
            'vol_2020_m3': ds['vol_2020_m3'],
        })
        .groupby(ds[class_coord])
        .sum(dim='rgi_id', skipna=True)
        .to_dataframe()
        .reset_index()
    )

    static_by_class['n_glaciers'] = static_by_class['n_glaciers'].astype(int)

    steady_by_class = (
        xr.Dataset({
            'area_steady': ds['area_steady'],
            'volume_steady': ds['volume_steady'],
        })
        .groupby(ds[class_coord])
        .sum(dim='rgi_id', skipna=True)
        .to_dataframe()
        .reset_index()
    )

    df = steady_by_class.merge(
        static_by_class,
        on=class_coord,
        how='left',
    )

    df['mass_remaining'] = (df['volume_steady'] / df['vol_2020_m3']) * 100

    cols = [
        class_coord,
        'n_glaciers',
        'rgi_area_km2',
        'vol_2020_m3',
        'experiment',
        'gcm',
        'period_scenario',
        'temp_ch_ipcc',
        'area_steady',
        'volume_steady',
        'mass_remaining',
    ]

    return df[cols]


def save_mt_class_csv(ds, class_name, outpath, tag):
    df = summarize_by_class(ds, 'region_class')
    if df.empty:
        print('Skipped empty class:', class_name)
        return

    output_csv = outpath / f'PyGEM_glacier_mass_{class_name}{tag}.csv'
    df.to_csv(output_csv, index=False)

#### Marine-terminating glaciers with frontal ablation observations

In [7]:
class_name = 'MT_19_with_obs'
mask = (
    (ds['region'] == 19)
    & (ds['is_tidewater'] == 1)
    & (ds['rgi_id'].isin(ind['RGIId']))
)
ds_sub = ds.where(mask, drop=True)

MT_class = np.repeat(class_name, ds_sub.sizes['rgi_id'])

ds_sub = ds_sub.assign_coords(region_class=('rgi_id', MT_class))

In [8]:
save_mt_class_csv(ds_sub, class_name, outpath, '')

In [9]:
run_lowess_from_csv(
    outpath / f'PyGEM_glacier_mass_{class_name}.csv',
    outpath / f'PyGEM_glacier_mass_{class_name}_{y_col}_lowess_fit.csv',
    trials_output_csv=None,
    x_col='temp_ch_ipcc',
    y_col=y_col,
    qs=None,
    preliminary_num_fits=500,
    final_num_fits=2000,
    robust_iters=2,
)

quantiles,temp_ch_ipcc,0.05,0.17,0.25,0.5,0.75,0.83,0.95,frac,it,N,final_num_fits,fit_opt,y,source_input,x_col,y_col
0,-0.10,76.437895,97.037266,97.290946,102.404605,111.592119,111.955554,122.146363,0.24,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
1,-0.05,74.216234,93.624850,94.382492,100.219098,109.417866,109.837297,121.115505,0.24,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
2,0.00,71.969486,90.373312,91.591724,98.095418,107.235389,107.711828,120.026804,0.24,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
3,0.05,69.717757,87.276154,88.918668,96.033291,105.050639,105.581482,118.884814,0.24,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
4,0.10,67.484286,84.323700,86.359343,94.027295,102.871162,103.446940,117.680080,0.24,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,6.65,1.223053,1.292610,1.623829,2.574098,3.286000,5.329621,9.342362,0.24,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
136,6.70,1.159740,1.299507,1.563919,2.451232,3.031252,4.968793,8.882750,0.24,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
137,6.75,1.096063,1.306100,1.505077,2.335860,2.782755,4.616167,8.457816,0.24,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
138,6.80,1.032030,1.312329,1.448153,2.225606,2.538427,4.275385,8.067976,0.24,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining


#### Marine-terminating glaciers missing frontal ablation observations
#### median_a

In [10]:
class_name = 'MT_19_missing_obs'
mask = (
    (ds['region'] == 19)
    & (ds['is_tidewater'] == 1)
    & (ds['rgi_id'].isin(missing['RGIId']))
)
ds_sub = ds.where(mask, drop=True)

MT_class = np.repeat(class_name, ds_sub.sizes['rgi_id'])

ds_sub = ds_sub.assign_coords(region_class=('rgi_id', MT_class))

In [11]:
save_mt_class_csv(ds_sub, class_name, outpath, f'_{tag}')

In [12]:
run_lowess_from_csv(
    outpath / f'PyGEM_glacier_mass_{class_name}_{tag}.csv',
    outpath / f'PyGEM_glacier_mass_{class_name}_{y_col}_{tag}_lowess_fit.csv',
    trials_output_csv=None,
    x_col='temp_ch_ipcc',
    y_col=y_col,
    qs=None,
    preliminary_num_fits=500,
    final_num_fits=2000,
    robust_iters=2,
)

quantiles,temp_ch_ipcc,0.05,0.17,0.25,0.5,0.75,0.83,0.95,frac,it,N,final_num_fits,fit_opt,y,source_input,x_col,y_col
0,-0.10,87.651733,101.809536,109.986848,115.996890,124.330741,125.385285,142.747236,0.25,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
1,-0.05,84.679763,99.288899,107.001331,113.563303,122.168365,123.472265,141.767917,0.25,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
2,0.00,81.759185,96.871919,104.137145,111.168123,119.996819,121.532066,140.610534,0.25,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
3,0.05,78.914354,94.545541,101.388705,108.812818,117.818080,119.564431,139.275677,0.25,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
4,0.10,76.167649,92.288515,98.739465,106.495072,115.632045,117.569214,137.762012,0.25,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,6.65,0.000000,0.000000,0.000000,0.535816,2.380902,3.334032,4.727993,0.25,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
136,6.70,0.000000,0.000000,0.000000,0.435230,2.182596,2.981009,4.545580,0.25,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
137,6.75,0.000000,0.000000,0.000000,0.335096,1.985085,2.610858,4.359717,0.25,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
138,6.80,0.000000,0.000000,0.000000,0.235362,1.787923,2.225437,4.170652,0.25,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining


#### Marine-terminating glaciers missing frontal ablation observations
#### median_k

In [17]:
tag = 'median_k'
ds = xr.open_dataset(outpath / f'PyGEM_global_glacier_stats_{tag}.nc')

ERROR! Session/line number was not unique in database. History logging moved to new session 3132


In [18]:
class_name = 'MT_19_missing_obs'
mask = (
    (ds['region'] == 19)
    & (ds['is_tidewater'] == 1)
    & (ds['rgi_id'].isin(missing['RGIId']))
)
ds_sub = ds.where(mask, drop=True)

MT_class = np.repeat(class_name, ds_sub.sizes['rgi_id'])

ds_sub = ds_sub.assign_coords(region_class=('rgi_id', MT_class))

In [19]:
save_mt_class_csv(ds_sub, class_name, outpath, f'_{tag}')

In [20]:
run_lowess_from_csv(
    outpath / f'PyGEM_glacier_mass_{class_name}_{tag}.csv',
    outpath / f'PyGEM_glacier_mass_{class_name}_{y_col}_{tag}_lowess_fit.csv',
    trials_output_csv=None,
    x_col='temp_ch_ipcc',
    y_col=y_col,
    qs=None,
    preliminary_num_fits=500,
    final_num_fits=2000,
    robust_iters=2,
)

quantiles,temp_ch_ipcc,0.05,0.17,0.25,0.5,0.75,0.83,0.95,frac,it,N,final_num_fits,fit_opt,y,source_input,x_col,y_col
0,-0.10,100.783646,109.355728,109.733714,115.180511,117.255930,119.696375,127.782027,0.34,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
1,-0.05,99.711293,108.507358,109.097774,114.287096,116.684260,118.924566,126.968889,0.34,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
2,0.00,98.690741,107.639000,108.445863,113.398383,116.080825,118.137497,126.101037,0.34,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
3,0.05,97.706951,106.757161,107.775438,112.513896,115.440505,117.335562,125.182609,0.34,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
4,0.10,96.744288,105.870632,107.091999,111.628374,114.758568,116.518555,124.220230,0.34,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,6.65,7.312523,6.144986,8.025909,10.799579,11.386599,11.517367,17.443740,0.34,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
136,6.70,7.164396,5.985875,7.840781,10.565489,10.952528,11.053303,16.694866,0.34,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
137,6.75,7.011450,5.826182,7.661850,10.333269,10.520670,10.589944,15.941914,0.34,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
138,6.80,6.854637,5.665093,7.489759,10.102219,10.090780,10.127156,15.180580,0.34,2,500,2000,lowess_fit,NaN,/g/data/rd53/wy2165/disequilibrium/pygem_oggm/...,temp_ch_ipcc,mass_remaining
